In [35]:
import pandas as pd
import sys
import numpy as np
import warnings
import os

sys.path.append("/Users/ejowik001/Desktop/Github/Nowcasting/kedro/refinery/dependencies/")

In [36]:
from estimation import (
    cast_to_base_unit,
    estimate_automl,
    calculate_contributions,
    calculate_conf_bounds,
)

from plots import plot_prediction
from retransform_prediction import retransform_
from retransform_data import retransform_data
from utils_ import cast_spec_to_dict, _convert_to_datetime

In [37]:
from dateutil.relativedelta import relativedelta

In [38]:
EPSILON = 1e-10

def rrse(actual: np.ndarray, predicted: np.ndarray, benchmark: np.ndarray=None):
    """ Root Relative Squared Error """
    return np.sqrt(
        np.sum(np.square(actual - predicted))
        / np.sum(np.square(actual - benchmark))
    )

def _error(actual: np.ndarray, predicted: np.ndarray):
    """ Simple error """
    return actual - predicted

def _percentage_error(actual: np.ndarray, predicted: np.ndarray):
    """
    Percentage error

    Note: result is NOT multiplied by 100
    """
    return _error(actual, predicted) / (actual + EPSILON)

def mape(actual: np.ndarray, predicted: np.ndarray):
    """
    Mean Absolute Percentage Error

    Note: result is NOT multiplied by 100
    """
    return np.mean(np.abs(_percentage_error(actual, predicted)))

def mse(actual: np.ndarray, predicted: np.ndarray):
    """ Mean Squared Error """
    return np.mean(np.square(_error(actual, predicted)))


def rmse(actual: np.ndarray, predicted: np.ndarray):
    """ Root Mean Squared Error """
    return np.sqrt(mse(actual, predicted))


In [39]:
def assign_weights(s):
    if (s['directional_accuracy'] == -1) and (s['within_cbounds'] == -1):
        return 2
    elif (s['directional_accuracy'] == -1) and (s['within_cbounds'] == 1):
        return 1.75
    elif (s['directional_accuracy'] == 1) and (s['within_cbounds'] == -1):
        return 1.25
    elif (s['directional_accuracy'] == 1) and (s['within_cbounds'] == 1):
        return 1
    else: return np.infty

In [40]:
confidence_bounds_func = lambda row: row['lower']<=row['y_pred']<=row['upper']

In [41]:
def wdmpe(predicted, actual):
    actual_diff = actual.sort_index().diff()
    actual_signs = np.sign(actual_diff)
    predicted_diff = predicted.sort_index().diff()
    predicted_signs = np.sign(predicted_diff)

    resid = predicted-actual

    dir_acc = list(actual_signs * predicted_signs)

    resid_mean = resid.expanding(1).mean()
    resid_std = resid.expanding(2).std().fillna(0)

    lower = actual-resid_std
    upper = actual+resid_std

    df = pd.DataFrame({
        "directional_accuracy": dir_acc,
        "lower": lower,
        "upper": upper,
        "y_pred": predicted
    }).iloc[1:, :]
    df['within_cbounds'] = df.apply(confidence_bounds_func, axis=1).astype(int).replace({0: -1})
    df['percentage_error'] = resid / actual

    df['weights'] = df.apply(assign_weights, axis=1)
    df["weighted_percentage_error"] = df['weights'] * df['percentage_error']
    return df["weighted_percentage_error"].mean()


In [42]:
# def cast_to_base_unit(ds, model_result, spec, series_name):
#     Spec = cast_spec_to_dict(spec.loc[spec["seriesid"] == series_name])

#     ## Retransform
#     ds = _convert_to_datetime(ds, ['ReferenceDate'])

#     dsrc = ds.set_index('ReferenceDate')

#     # def retransform_prediction(transf_series, base_series, Spec, series_name):
#     base_series = dsrc[series_name]
#     header = [series_name]

#     backcast = model_result['predictions']['backcast']
#     forecast = pd.Series(
#         model_result["predictions"]["forecast"],
#         index=[model_result["predictions"]["reference_date"]]
#         )

#     transf_pred = pd.concat([backcast, forecast])
#     transf_pred.index = pd.to_datetime(transf_pred.index)

#     transf_series = model_result["actual"]

#     Time = np.sort(np.unique(np.concatenate((base_series.index.date, transf_pred.index.date))))
#     cutoff_date = transf_pred.index.min().date()

#     Z = base_series.reindex(Time).to_numpy().reshape(-1,1)

#     Yhat = transf_pred.reindex(Time).to_numpy().reshape(-1,1)
#     Y = transf_series.reindex(Time).to_numpy().reshape(-1,1)

#     Rhat = retransform_(X=Yhat, Z=Z, Time=Time, Spec=Spec, header=header, cutoff_date=cutoff_date)
#     R = retransform_data(X=Y, Z=Z, Time=Time, Spec=Spec, header=header, cutoff_date=cutoff_date)

#     return Rhat, R, Time, cutoff_date

In [43]:
series_name = "PCEC96"
reference_date = "2024-08-01"
n_periods = 60
plt_out_dir = "../data/07_model_output/"

In [44]:
ds = pd.read_parquet("../data/04_feature/selected_series.parquet")


In [58]:
spec = pd.read_csv("../data/02_intermediate/variable.csv")
ds_base = pd.read_parquet("../data/02_intermediate/non_transformed_data.parquet")

# Example usage
model_result = estimate_automl(
    ds=ds,
    ds_base=ds_base,
    spec=spec,
    ref_date_col="ReferenceDate",
    series_name=series_name,
    reference_date=reference_date,
    n_periods=n_periods,
)


# reference_date = pd.to_datetime(parameters["ref_date"]).date()
reference_date = pd.to_datetime(reference_date)
lag_date = reference_date - relativedelta(months=1)
# lag = model_result["pred_"]["backcast"].loc[(reference_date-relativedelta(months=1)).strftime("%Y-%m-%d")]
pred = model_result["pred_"]["forecast"]
coef_ = model_result["coef_"]
values = model_result["values"]

# Print the best model's details
print("============ Model Details ============")
print(f"Model                     : {model_result['best_model']}")
print(f"Reference Date            : {reference_date}")
print(f"Forecast                  : {pred:.4f}")
print(f"R-Squared (R²)            : {model_result['r_squared']:.4f}")
print(f"Mean Absolute Percentage Error (MAPE): {model_result['mape']:.2f}%")
print(f"Root Mean Square Error (RMSE) : {model_result['rmse']:.4f}")

# print(calculate_contributions(coef_, pred, lag, values))

formula = spec.loc[spec["seriesid"] == series_name][
    "transformation"
].item()
unit = spec.loc[spec["seriesid"] == series_name]["units"].item()
dt = model_result["pred_"]["backcast"].index

pred = model_result["pred_"]["backcast"]
actual = model_result["actual"].loc[dt]
bounds = calculate_conf_bounds(pred, actual)

# plot_prediction(
#     dt=dt,
#     y_pred=pred,
#     y_actual=actual,
#     mode="lines+markers",
#     lower1=bounds["L1"],
#     upper1=bounds["U1"],
#     lower2=bounds["L2"],
#     upper2=bounds["U2"],
#     title=f'Series: {series_name}, Reference Date: {reference_date}, Unit: {unit} {formula}',
#     # plt_out_path=os.path.join(parameters["fig_out_dir"], f'{model_result["best_model"]}_{datetime.now().strftime("%Y%m%d%H%M%S")}_pva.png')
# )

transf_pred = pd.concat(
    [
        model_result["pred_"]["backcast"],
        pd.Series(
            model_result["pred_"]["forecast"],
            index=[pd.to_datetime(model_result["pred_"]["reference_date"])],
        ),
    ]
)
transf_actual = model_result["actual"]

# Retransform forecast
Rhat, Time, cutoff_date = cast_to_base_unit(
    ds_base, spec, series_name, transf_pred, dtype="pred"
)
R, _, _ = cast_to_base_unit(
    ds_base, spec, series_name, transf_actual, dtype="actual"
)

# TBC
header = [series_name]
Rhat_df = pd.DataFrame(Rhat, columns=header, index=Time)
R_df = pd.DataFrame(R, columns=header, index=Time)

retr_forecast = Rhat_df.loc[reference_date.date()].item()
retr_actual = R_df.loc[reference_date.date()].item()
retr_lag = R_df.loc[lag_date.date()].item()

contributions = calculate_contributions(coef_, retr_forecast, retr_lag, values)

print("\n============ Forecast vs Actual ============")
print(f"Reference Date            : {reference_date}")
print(f"Retransformed Forecast    : {retr_forecast:,.2f}")
print(f"Actual Release            : {retr_actual:,.2f}")
print(
    f"Percentage Error (Level)  : {(retr_forecast - retr_actual) / retr_actual:.2%}"
)

bounds_level = {}
for key, value in bounds.items():
    data, dt_, _ = cast_to_base_unit(
        ds_base, spec, series_name, value, dtype="pred"
    )
    tmp = pd.Series(data.reshape(1, -1)[0], index=dt_)
    bounds_level[key] = tmp.loc[dt]

# plot_prediction(
#     dt=dt,
#     y_pred=Rhat_df.loc[dt][series_name],
#     y_actual=R_df.loc[dt][series_name],
#     lower1=bounds_level["L1"],
#     upper1=bounds_level["U1"],
#     lower2=bounds_level["L2"],
#     upper2=bounds_level["U2"],
#     mode="lines+markers",
#     title=f'Series: {series_name}, Reference Date: {reference_date}, Unit: {unit}',
#     # plt_out_path=os.path.join(parameters["fig_out_dir"], f'{model_result["best_model"]}_{datetime.now().strftime("%Y%m%d%H%M%S")}_bpva.png')
# )

============ Model Details ============
Model                     : Ridge
Reference Date            : 2024-08-01 00:00:00
Forecast                  : 0.1128
R-Squared (R²)            : 0.9012
Mean Absolute Percentage Error (MAPE): 4.34%
Root Mean Square Error (RMSE) : 1.7378

============ Forecast vs Actual ============
Reference Date            : 2024-08-01 00:00:00
Retransformed Forecast    : 15,888.21
Actual Release            : 16,089.70
Percentage Error (Level)  : -1.25%


In [ ]:
out_dir = "/Users/ejowik001/Desktop/Github/Nowcasting/kedro/refinery/data/07_model_output/"

# Save everything to an Excel file with multiple sheets
excel_file = os.path.join(out_dir, f'ml.xlsx')

with pd.ExcelWriter(excel_file, engine='xlsxwriter') as writer:
    # Save model details as a dataframe
    sheet1 = pd.DataFrame.from_dict({"Model Name": model_result["best_model"], "R-Squared": model_result["r_squared"], "MAPE": model_result["mape"], "RMSE": model_result["rmse"]}, orient="index", columns=["Value"]).reset_index().rename(columns={"index": "Banner"})
    sheet1.to_excel(writer, sheet_name="Model Details", index=False)
    # Save contributions
    contributions.to_excel(writer, sheet_name="Contributions")

    # Save forecast and actual values
    R_df = R_df.rename(columns={series_name: "Actual"})
    Rhat_df = Rhat_df.rename(columns={series_name: "Predicted"})
    sheet3 = pd.merge(R_df, Rhat_df, how="outer", left_index=True, right_index=True).reset_index().rename(columns={"index": "Reference Date"})
    sheet3.to_excel(writer, sheet_name="Forecast vs Actual", index=False)

print(f"Results saved to {excel_file}")

Results saved to /Users/ejowik001/Desktop/Github/Nowcasting/kedro/refinery/data/07_model_output/ml.xlsx
